# OCR POC — PaddleOCR Türkçe (Colab Pro GPU)

Goal: validate Turkish OCR on TBMM scanned yazılı soru önergesi PDFs.

Steps:
1. Install PaddleOCR + pdf2image
2. Download 1 sample PDF
3. Convert to images
4. OCR each page (GPU batch)
5. Measure confidence + Turkish accuracy

Decision criteria: avg confidence > 0.7 + manual readable → green light.

## 1. Setup

In [ ]:
!nvidia-smi

In [ ]:
!apt-get install -y poppler-utils -qq
# PaddleOCR 2.x pinned — 3.x retriever pulls langchain_text_splitters (broken transitive)
!pip install -q paddlepaddle-gpu "paddleocr<3.0" pdf2image pillow tqdm numpy

In [ ]:
import os, time, requests
from pathlib import Path
from pdf2image import convert_from_path
from paddleocr import PaddleOCR
import numpy as np

## 2. Download sample PDF

In [ ]:
# Sample known yazılı soru detail page (verified live)
DETAIL_URL = "https://www.tbmm.gov.tr/Denetim/Yazili-Soru-Onergesi-Detay/eea2e48d-d119-4df8-98cb-018d1179fa67"

headers = {"User-Agent": "Mozilla/5.0"}
html = requests.get(DETAIL_URL, headers=headers, timeout=30).text

import re
pdf_urls = re.findall(r'href="([^"]+\.pdf[^"]*)"', html)
print(f"Found {len(pdf_urls)} PDF link(s)")
for u in pdf_urls:
    print(" ", u)

from urllib.parse import urljoin
if not pdf_urls:
    raise RuntimeError("No PDF link in detail page — inspect HTML manually")

pdf_url = urljoin("https://www.tbmm.gov.tr", pdf_urls[0])
pdf_bytes = requests.get(pdf_url, headers=headers, timeout=60).content
Path("sample.pdf").write_bytes(pdf_bytes)
print(f"Saved sample.pdf ({len(pdf_bytes)/1024:.1f} KB)")

## 3. PDF → images

In [ ]:
pages = convert_from_path("sample.pdf", dpi=300)
print(f"{len(pages)} page(s) extracted")
pages[0]

## 4. OCR — PaddleOCR Türkçe

Note: PaddleOCR uses Latin-based model; Turkish characters supported via the multilingual `lang='latin'` or experimental `lang='tr'`. We'll try `latin` (most robust).

In [ ]:
ocr = PaddleOCR(use_angle_cls=True, lang='latin', use_gpu=True, show_log=False)

In [ ]:
results = []
for i, page in enumerate(pages):
    arr = np.array(page)
    t0 = time.time()
    res = ocr.ocr(arr, cls=True)
    dt = time.time() - t0
    lines = res[0] if res else []
    texts = [line[1][0] for line in lines]
    confs = [line[1][1] for line in lines]
    page_text = "\n".join(texts)
    avg_conf = float(np.mean(confs)) if confs else 0.0
    results.append({
        "page": i + 1,
        "n_lines": len(lines),
        "avg_confidence": avg_conf,
        "runtime_sec": dt,
        "text": page_text,
    })
    print(f"page {i+1}: {len(lines)} lines, conf={avg_conf:.3f}, {dt:.1f}s")

## 5. Inspect first page output

In [ ]:
print(results[0]["text"][:2000])

## 6. Decision report

In [ ]:
import pandas as pd
df = pd.DataFrame(results)
print(df[['page', 'n_lines', 'avg_confidence', 'runtime_sec']])

overall_conf = df['avg_confidence'].mean()
total_time = df['runtime_sec'].sum()
per_page = total_time / len(df)

print(f"\nOverall avg confidence: {overall_conf:.3f}")
print(f"Per-page runtime: {per_page:.2f}s")
print(f"Projected 20K PDF (avg 3pg) ETA: {20000 * 3 * per_page / 3600:.1f}h")

if overall_conf >= 0.70:
    print("✅ GREEN LIGHT — proceed with full pipeline")
elif overall_conf >= 0.55:
    print("⚠️ YELLOW — try lang='tr' or preprocess (denoise, deskew)")
else:
    print("❌ RED — switch to Tesseract tur model or Google Vision API")

## Next: scale to batch

If green light → `03_ocr_batch.ipynb`:
- Read Bronze PDFs from Drive
- Batch process (50 PDFs at once)
- Write Silver Delta table: `(guid, page_no, text, confidence, runtime)`
- Checkpoint every 500 to survive Colab session timeout